# MiniCPM-o Realtime Audio on Colab

Run this notebook on a Colab GPU runtime to deploy this repo's MiniCPM-o backend and expose `/v1/realtime` through a Cloudflare tunnel.

Recommended runtime: A100 or L4. Smaller free-tier GPUs may run out of memory while loading `openbmb/MiniCPM-o-4_5`.


## 1. Check GPU

In Colab, choose **Runtime > Change runtime type > GPU** before running the cells.

In [ ]:
!nvidia-smi

## 2. Clone Repo

This clones `erkamkavak/minicpm-o-agent` into `/content/minicpm-o-agent`. Change `BRANCH` only if you want a branch other than `main`.


In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/erkamkavak/minicpm-o-agent.git"
BRANCH = "main"
REPO_DIR = Path("/content/minicpm-o-agent")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "fetch", "origin"], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "checkout", BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "pull", "--ff-only"], cwd=REPO_DIR, check=True)

os.chdir(REPO_DIR)
print("Working directory:", Path.cwd())
!git status --short


## 3. Install Dependencies

The service launcher expects `.venv/base`, so this uses the repo's installer. Colab's Python image may be missing the matching `venv` package, so the cell installs it before running `install.sh`.


In [ ]:
%cd /content/minicpm-o-agent
!sudo apt-get update -qq
!PYVER=$(python3 -c "import sys; print(f'{sys.version_info.major}.{sys.version_info.minor}')") && (sudo apt-get install -y -qq python${PYVER}-venv || sudo apt-get install -y -qq python3-venv)
!PYTHON=python3 SKIP_FLASH_ATTN=1 bash install.sh


## 4. Optional Hugging Face Login

Run this only if model download fails because of rate limits, auth, or gated model access.

In [ ]:
# Optional. Uncomment if needed.
# from huggingface_hub import notebook_login
# notebook_login()

## 5. Configure Model And Service

This config downloads `openbmb/MiniCPM-o-4_5` from Hugging Face at first worker start and runs the gateway on local HTTP port `8006`.

In [ ]:
%%bash
cd /content/minicpm-o-agent
cp configs/config.example.json config.json
python - <<'PY'
import json
from pathlib import Path

path = Path("config.json")
cfg = json.loads(path.read_text())
cfg["model"]["model_path"] = "openbmb/MiniCPM-o-4_5"
cfg["model"]["pt_path"] = None
cfg["model"]["attn_implementation"] = "auto"
cfg["service"]["gateway_port"] = 8006
cfg["service"]["worker_base_port"] = 22400
cfg["service"]["compile"] = False
path.write_text(json.dumps(cfg, indent=4, ensure_ascii=False))
PY
cat config.json

## 6. Start Worker And Gateway

The first run can take a while because the worker downloads and loads the model. If this fails, inspect the log cells below.

In [ ]:
%cd /content/minicpm-o-agent
!CUDA_VISIBLE_DEVICES=0 bash start_all.sh --http

In [ ]:
!curl -s http://127.0.0.1:8006/health; echo
!curl -s http://127.0.0.1:8006/workers; echo

In [ ]:
# Run this if the service does not become healthy.
!tail -120 /content/minicpm-o-agent/tmp/worker_0.log || true
!tail -120 /content/minicpm-o-agent/tmp/gateway.log || true

## 7. Start Gateway Tunnel

This starts a Cloudflare tunnel for the gateway. Keep this runtime alive while testing. If the tunnel dies, rerun this cell and use the new URL.


In [ ]:
import os
import re
import subprocess
import time
from pathlib import Path

!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

def start_background(cmd, log_path, cwd=None):
    log = open(log_path, "w")
    return subprocess.Popen(cmd, cwd=cwd, stdout=log, stderr=subprocess.STDOUT, text=True)

def wait_for_tunnel_url(log_path, timeout_s=90):
    pattern = re.compile(r"https://[-a-zA-Z0-9.]+\\.trycloudflare\\.com")
    deadline = time.time() + timeout_s
    log_file = Path(log_path)
    while time.time() < deadline:
        text = log_file.read_text(errors="ignore") if log_file.exists() else ""
        match = pattern.search(text)
        if match:
            return match.group(0)
        time.sleep(1)
    raise RuntimeError(f"Timed out waiting for tunnel URL in {log_path}")

gateway_proc = start_background(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8006", "--no-autoupdate"],
    "/content/minicpmo-gateway-tunnel.log",
)
gateway_https_url = wait_for_tunnel_url("/content/minicpmo-gateway-tunnel.log")
gateway_wss_url = gateway_https_url.replace("https://", "wss://") + "/v1/realtime?mode=audio"

os.environ["MINICPMO_GATEWAY_HTTPS_URL"] = gateway_https_url
os.environ["MINICPMO_GATEWAY_WSS_URL"] = gateway_wss_url

print("Gateway health:", gateway_https_url + "/health")
print("Realtime WebSocket endpoint:", gateway_wss_url)
print("\nUse this WebSocket endpoint from your realtime audio client.")


## 8. Optional Probe

This sends the bundled test WAV through the Colab-hosted realtime endpoint without using your microphone.

In [ ]:
import os
import subprocess

gateway = os.environ.get("MINICPMO_GATEWAY_HTTPS_URL")
if not gateway:
    raise RuntimeError("Run the tunnel cell first.")

cmd = [
    "/content/minicpm-o-agent/.venv/base/bin/python",
    "audio_probe.py",
    "--url", gateway,
    "--input-wav", "assets/test.wav",
    "--region", "colab",
    "--pretty-json",
    "--max-session-s", "90",
]
subprocess.run(cmd, cwd="/content/minicpm-o-agent/examples/realtime", check=False)

## 9. Stop Everything

Run this when you are done.


In [ ]:
!kill $(cat /content/minicpm-o-agent/tmp/*.pid 2>/dev/null) 2>/dev/null || true
!pkill -f "cloudflared tunnel --url" 2>/dev/null || true
